# Senate Financial Disclosures — Collect + OCR pipeline

A **two-phase, resumable** notebook for Google Colab.

- **Phase 1 — Collect.** Politely sweeps the public U.S. Senate eFD search
  (`efdsearch.senate.gov`), filters to **Annual** and **Periodic Transaction**
  reports for **current + former senators**, and downloads each report to Google
  Drive in its native format (electronic reports as HTML, paper filings as scanned
  PDF/images). Writes a **manifest** indexing everything.
- **Phase 2 — OCR.** Runs **Gemma 4 12B** (multimodal) over the *scanned* subset to
  turn page images into text, writing structured JSON per document.

### How to run it
1. **Phase 1** needs only the network — run it on a **CPU** runtime
   (*Runtime → Change runtime type → CPU*) so it spends **zero** GPU compute units.
2. When Phase 1 is done, switch to an **L4 GPU** (*Runtime → Change runtime type → L4*)
   and run **Phase 2**.
3. Both phases **checkpoint after every item** and **skip work already done**, so if
   Colab disconnects overnight you just re-run the phase and it resumes. Leaving the
   browser open (and your computer awake) keeps a Colab **Pro** session alive; a mid-run
   disconnect costs you at most one item.

### A note on the source & terms
These are public records filed under the Ethics in Government Act. This notebook
accepts the site's terms-of-service gate programmatically and paces itself with delays
to be gentle on the server. The disclosures may not be used for commercial
exploitation, credit-rating, or solicitation — using them for personal research/analysis
is fine. You are responsible for your own use.

> **First-run validation:** this notebook was authored without live access to the site,
> so the exact request payload and the paper-viewer structure are built from known-good
> open-source scrapers. **Run the "Smoke test" cell first** — it proves the handshake and
> payload return real rows and shows you one electronic and one paper report before any
> bulk run. If the site's field names have drifted, you'll adjust one dict, not the whole
> notebook.

## 0 · Config — edit these, then run top to bottom

In [1]:
# ===================== EDIT ME =====================

# --- Where everything is saved (on mounted Google Drive) ---
LOCAL_DRIVE_BASE   = "/content/sample_data/Senate"   # corpus + manifest + OCR output live here
CLOUD_DRIVE_BASE   = "<REDACTED-DRIVE-FOLDER-ID>"           # folder ID in Google Drive
# --- Collection scope (Phase 1) ---
ROOT              = "https://efdsearch.senate.gov"
# Report-type labels to KEEP (matched case-insensitively against each row's report title):
KEEP_REPORT_TYPES = ("Annual Report", "Periodic Transaction")
# Filer types to KEEP / DROP (matched against the row's office label):
KEEP_FILER_HINT   = "senator"     # keeps current AND former senators
DROP_FILER_HINT   = "candidate"   # drops candidate filings
DATE_FROM         = "01/01/2012"  # site coverage begins 2012 (MM/DD/YYYY)
DATE_TO           = ""            # blank = up to today

# --- Politeness / robustness ---
REQUEST_DELAY_SECS = 2.0          # pause between requests to the site
PAGE_SIZE          = 100          # rows per listing request
MAX_RETRIES        = 5
BACKOFF_BASE_SECS  = 3.0          # exponential backoff on 429/5xx
# The site sits behind an edge/WAF that returns 403 to non-browser User-Agents, so we
# must present as a normal browser. We stay polite via the delays above and keep an
# identifying contact in the 'From' header.
CONTACT_EMAIL = "<REDACTED-EMAIL>"
USER_AGENT = ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
              "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36")
BROWSER_HEADERS = {
    "User-Agent": USER_AGENT,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Upgrade-Insecure-Requests": "1",
    "Connection": "keep-alive",
    "From": CONTACT_EMAIL,
}

# --- OCR (Phase 2) ---
MODEL_ID       = "google/gemma-4-12b-it"  # gated: accept license at hf.co/google/gemma-4-12b-it
LOAD_IN_4BIT   = True     # fits comfortably on an L4 (24 GB); set False for bf16 on A100
PDF_RENDER_DPI = 200      # higher -> better small-text OCR, more memory/time (try 150-250)
MAX_NEW_TOKENS = 4096
OCR_PROMPT = (
    "You are transcribing a U.S. Senate financial disclosure page. "
    "Transcribe ALL text in this image verbatim. Preserve any table as rows with "
    "columns separated by ' | '. Include every transaction line, date, owner, "
    "asset name/ticker, transaction type (purchase/sale/exchange), and amount range "
    "exactly as printed. Do not summarize, reorder, interpret, or omit anything. "
    "If a region is illegible, write [illegible]."
)
# ===================================================

import os
CORPUS_DIR   = os.path.join(LOCAL_DRIVE_BASE, "reports")      # downloaded reports
OCR_DIR      = os.path.join(LOCAL_DRIVE_BASE, "ocr")          # per-document OCR JSON
MANIFEST_CSV = os.path.join(LOCAL_DRIVE_BASE, "manifest.csv") # index of everything
print("Corpus  ->", CORPUS_DIR)
print("OCR out ->", OCR_DIR)
print("Manifest->", MANIFEST_CSV)

Corpus  -> /content/sample_data/Senate/reports
OCR out -> /content/sample_data/Senate/ocr
Manifest-> /content/sample_data/Senate/manifest.csv


## 1 · Mount Google Drive & create folders

In [2]:
# from google.colab import drive
# drive.mount("/content/drive")

# --- One-time install (google-api-python-client is usually preinstalled in Colab) ---
!pip install -q google-api-python-client google-auth

import json, io, os
from google.colab import userdata
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# --- Authenticate as the service account ---
# Read-only scope. Use ".../auth/drive" (no .readonly) if you need to write back.
#SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]
SCOPES = ["https://www.googleapis.com/auth/drive"]

key_info = json.loads(userdata.get("agent_key"))          # pulled from Colab Secrets
# print(key_info)
creds = service_account.Credentials.from_service_account_info(
    key_info, scopes=SCOPES
)
drive = build("drive", "v3", credentials=creds)

# --- Point at your shared folder (from its Drive URL) ---
#  Replaced with CLOUD_DRIVE_BASE above
# FOLDER_ID = "1Nhw8-CkfhELEy-Km3RVcQxEwT6UhC124"

# --- List files in that folder ---
results = drive.files().list(
    q=f"'{CLOUD_DRIVE_BASE}' in parents and trashed = false",
    fields="files(id, name, mimeType, size)",
    pageSize=1000,
    supportsAllDrives=True,            # needed if the folder lives in a Shared Drive
    includeItemsFromAllDrives=True,
).execute()

files = results.get("files", [])
for f in files:
    print(f["name"], "—", f["id"], f"({f.get('size','?')} bytes)")

for d in (LOCAL_DRIVE_BASE, CORPUS_DIR, OCR_DIR):
    os.makedirs(d, exist_ok=True)
print("Ready.")

test — 1OtPodaDh9vPLc6bHlb_UcraQiwl9oz8G (? bytes)
mnist_train_small.csv — 1XXDZ0fKZRTW-hyVziYe-faAG9P8wmvBn (36523880 bytes)
mnist_test.csv — 12R-5l6v4kHcGnm2zVoLnn0lBLGmxNrbD (18289443 bytes)
california_housing_train.csv — 1TDnW76BRCDLL0tUh7vLnLUK-1j43AQ86 (1706430 bytes)
california_housing_test.csv — 1mh7Cg-0Etg6r2NfXSr4ZFfY-rESS8reE (301141 bytes)
anscombe.json — 1iBRDGCl_EJobBOHhN04-mJqD_nHcgwHk (1697 bytes)
Senate — 1G0nTAxxpv3GHrubKQ-AlSytKLDsRWZOj (? bytes)
README.md — 1caiWrrUga3xfk1zYXj3rN1bN6bDhWxoi (962 bytes)
Ready.


In [3]:
# sync up and sync down helper functions
import io, os, hashlib
from datetime import datetime, timezone

from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
from googleapiclient.errors import HttpError

GOOGLE_NATIVE_EXPORTS = {
    "application/vnd.google-apps.document": (
        "application/vnd.openxmlformats-officedocument.wordprocessingml.document", ".docx"),
    "application/vnd.google-apps.spreadsheet": (
        "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet", ".xlsx"),
    "application/vnd.google-apps.presentation": (
        "application/vnd.openxmlformats-officedocument.presentationml.presentation", ".pptx"),
    "application/vnd.google-apps.drawing": ("image/png", ".png"),
    "application/vnd.google-apps.script": ("application/vnd.google-apps.script+json", ".json"),
}

FOLDER_MIME = "application/vnd.google-apps.folder"

# Fields we request on every file listing. Keep in sync with anything read below.
_FILE_FIELDS = "id, name, mimeType, size, modifiedTime, md5Checksum"


# --------------------------------------------------------------------------- #
# Listing helpers
# --------------------------------------------------------------------------- #

def list_folder(drive, folder_id):
    """Return direct children (files and subfolders) of a Drive folder.

    Handles pagination and Shared Drives. Returns a list of dicts with the
    fields in _FILE_FIELDS.
    """
    items = []
    page_token = None
    while True:
        resp = drive.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields=f"nextPageToken, files({_FILE_FIELDS})",
            pageSize=1000,
            pageToken=page_token,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        ).execute()
        items.extend(resp.get("files", []))
        page_token = resp.get("nextPageToken")
        if not page_token:
            break
    return items


def _find_child(drive, parent_id, name, mime_type=None):
    """Find a direct child by name (and optionally mimeType). Returns the file
    dict or None. Used to decide create-vs-update when pushing up."""
    # Escape single quotes in the name for the query string.
    safe_name = name.replace("\\", "\\\\").replace("'", "\\'")
    q = f"name = '{safe_name}' and '{parent_id}' in parents and trashed = false"
    if mime_type:
        q += f" and mimeType = '{mime_type}'"
    resp = drive.files().list(
        q=q,
        fields=f"files({_FILE_FIELDS})",
        pageSize=10,
        supportsAllDrives=True,
        includeItemsFromAllDrives=True,
    ).execute()
    files = resp.get("files", [])
    return files[0] if files else None


# --------------------------------------------------------------------------- #
# Download side
# --------------------------------------------------------------------------- #

def _local_matches_remote(local_path, remote):
    """Cheap skip check: does the local file already match the remote?

    Prefers md5 when Drive provides it (only for binary/blob files, not Google
    native docs), otherwise falls back to size. Not perfect, but avoids
    re-downloading unchanged blobs on repeated syncs.
    """
    if not os.path.exists(local_path):
        return False
    remote_md5 = remote.get("md5Checksum")
    if remote_md5:
        h = hashlib.md5()
        with open(local_path, "rb") as fh:
            for chunk in iter(lambda: fh.read(1024 * 1024), b""):
                h.update(chunk)
        return h.hexdigest() == remote_md5
    # No md5 (e.g. exported Google Doc) — fall back to size when present.
    remote_size = remote.get("size")
    if remote_size is not None:
        return os.path.getsize(local_path) == int(remote_size)
    return False  # can't tell → re-download to be safe


def _download_blob(drive, file_id, dest_path):
    request = drive.files().get_media(fileId=file_id, supportsAllDrives=True)
    with io.FileIO(dest_path, "wb") as fh:
        downloader = MediaIoBaseDownload(fh, request, chunksize=8 * 1024 * 1024)
        done = False
        while not done:
            _, done = downloader.next_chunk()


def _export_native(drive, file_id, export_mime, dest_path):
    request = drive.files().export_media(fileId=file_id, mimeType=export_mime)
    with io.FileIO(dest_path, "wb") as fh:
        downloader = MediaIoBaseDownload(fh, request, chunksize=8 * 1024 * 1024)
        done = False
        while not done:
            _, done = downloader.next_chunk()


# --------------------------------------------------------------------------- #
# Parallel download (Drive -> local)
# --------------------------------------------------------------------------- #
# The corpus is ~10k files / ~2 GB, so the download is the real wall-time sink. It's
# pure network I/O against Drive (no Senate involvement), so we fan the blob downloads
# out across a thread pool. As on the upload side: enumerate + make local dirs SERIALLY
# first (no shared-state races), then parallelize only the byte transfers, each worker
# using its OWN Drive client (googleapiclient / httplib2 aren't thread-safe to share).

import concurrent.futures, threading, random, time

DRIVE_DOWNLOAD_WORKERS = 8      # shares one service-account quota bucket; ~8 is the sweet spot
_DL_MAX_TRIES = 6

_dl_thread_local = threading.local()
def _dl_thread_drive():
    d = getattr(_dl_thread_local, "drive", None)
    if d is None:
        d = build("drive", "v3", credentials=service_account.Credentials
                  .from_service_account_info(key_info, scopes=SCOPES))
        _dl_thread_local.drive = d
    return d

def _dl_with_backoff(fn, *args, **kwargs):
    """Run a download/export with exponential backoff + jitter on transient Drive errors."""
    for attempt in range(_DL_MAX_TRIES):
        try:
            return fn(*args, **kwargs)
        except HttpError as e:
            status = getattr(e.resp, "status", None)
            reason = str(e)
            retryable = status in (429, 500, 502, 503, 504) or (
                status == 403 and ("ateLimit" in reason or "uotaExceeded" in reason))
            if not retryable or attempt == _DL_MAX_TRIES - 1:
                raise
            time.sleep(min(2 ** attempt + random.random(), 32))

def _enumerate_downloads(drive, folder_id, local_dir):
    """Recursively walk the Drive tree (metadata only) and return a flat worklist of
    items to fetch, plus the set of local directories that must exist. Applies the SAME
    skip logic and Google-native export mapping as the serial version."""
    work = []          # list of dicts: file_id, dest_path, export_mime (or None)
    dirs = {local_dir}
    skipped = 0
    stack = [(folder_id, local_dir)]
    while stack:
        fid, ldir = stack.pop()
        dirs.add(ldir)
        for item in list_folder(drive, fid):
            name, mime = item["name"], item["mimeType"]
            if mime == FOLDER_MIME:
                stack.append((item["id"], os.path.join(ldir, name)))
                continue
            if mime in GOOGLE_NATIVE_EXPORTS:
                export_mime, ext = GOOGLE_NATIVE_EXPORTS[mime]
                dest_name = name if name.lower().endswith(ext) else name + ext
                dest_path = os.path.join(ldir, dest_name)
                if _local_matches_remote(dest_path, item):
                    skipped += 1; continue
                work.append({"file_id": item["id"], "dest_path": dest_path,
                             "export_mime": export_mime, "name": dest_name})
            else:
                dest_path = os.path.join(ldir, name)
                if _local_matches_remote(dest_path, item):
                    skipped += 1; continue
                work.append({"file_id": item["id"], "dest_path": dest_path,
                             "export_mime": None, "name": name})
    return work, dirs, skipped

def _fetch_one(job):
    """Worker body: download or export a single file using this thread's own client."""
    drv = _dl_thread_drive()
    if job["export_mime"]:
        req = drv.files().export_media(fileId=job["file_id"], mimeType=job["export_mime"])
        kind = "exported"
    else:
        req = drv.files().get_media(fileId=job["file_id"], supportsAllDrives=True)
        kind = "downloaded"
    # download to a temp path, then atomically move into place, so an interrupted
    # transfer never leaves a half-written file that the skip-check would trust.
    tmp = job["dest_path"] + ".part"
    with io.FileIO(tmp, "wb") as fh:
        downloader = MediaIoBaseDownload(fh, req, chunksize=8 * 1024 * 1024)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    os.replace(tmp, job["dest_path"])
    return kind

def sync_drive_folder_to_local(drive, folder_id, local_dir, verbose=True):
    """Recursively download a Drive folder into local_dir, IN PARALLEL.

    Same behavior as before (mirrors subfolders, exports Google-native docs, skips
    files whose local copy already matches by md5/size) -- but blob transfers run across
    a thread pool. Returns {"downloaded", "skipped", "exported", "errors"}.
    """
    os.makedirs(local_dir, exist_ok=True)
    if verbose:
        print("Enumerating Drive tree...")
    work, dirs, skipped = _enumerate_downloads(drive, folder_id, local_dir)
    for d in sorted(dirs):                       # create all local dirs up front (serial)
        os.makedirs(d, exist_ok=True)
    summary = {"downloaded": 0, "skipped": skipped, "exported": 0, "errors": 0}
    if verbose:
        print(f"{len(work)} files to fetch, {skipped} already up-to-date. "
              f"Downloading with {DRIVE_DOWNLOAD_WORKERS} workers...")

    done_n = 0
    with concurrent.futures.ThreadPoolExecutor(max_workers=DRIVE_DOWNLOAD_WORKERS) as ex:
        futs = {ex.submit(_dl_with_backoff, _fetch_one, job): job for job in work}
        for fut in concurrent.futures.as_completed(futs):
            job = futs[fut]
            try:
                kind = fut.result()
                summary["exported" if kind == "exported" else "downloaded"] += 1
            except Exception as e:
                summary["errors"] += 1
                print(f"  ERROR on {job['name']}: {e}")
                # clean up any partial file so a re-run retries it
                try: os.remove(job["dest_path"] + ".part")
                except OSError: pass
            done_n += 1
            if verbose and done_n % 250 == 0:
                print(f"  {done_n}/{len(work)} fetched")
    if verbose:
        print(f"Download complete: {summary}")
    return summary

def _sync_drive_folder_to_local_serial(drive, folder_id, local_dir, verbose=True):
    """Recursively download a Drive folder into local_dir.

    - Recreates the subfolder structure under local_dir.
    - Google-native files (Docs/Sheets/Slides) are exported to Office formats
      with an added extension (report -> report.docx).
    - Skips files whose local copy already matches (md5 or size).

    Returns a summary dict: {"downloaded": n, "skipped": n, "exported": n}.
    """
    os.makedirs(local_dir, exist_ok=True)
    summary = {"downloaded": 0, "skipped": 0, "exported": 0, "errors": 0}

    for item in list_folder(drive, folder_id):
        name = item["name"]
        mime = item["mimeType"]

        if mime == FOLDER_MIME:
            # Recurse into subfolder.
            sub_local = os.path.join(local_dir, name)
            sub_summary = sync_drive_folder_to_local(
                drive, item["id"], sub_local, verbose=verbose)
            for k in summary:
                summary[k] += sub_summary.get(k, 0)
            continue

        try:
            if mime in GOOGLE_NATIVE_EXPORTS:
                export_mime, ext = GOOGLE_NATIVE_EXPORTS[mime]
                # Only add the extension if the name doesn't already carry it.
                dest_name = name if name.lower().endswith(ext) else name + ext
                dest_path = os.path.join(local_dir, dest_name)
                if _local_matches_remote(dest_path, item):
                    summary["skipped"] += 1
                    if verbose:
                        print(f"  skip (native)  {dest_name}")
                    continue
                if verbose:
                    print(f"  export         {dest_name}")
                _export_native(drive, item["id"], export_mime, dest_path)
                summary["exported"] += 1
            else:
                dest_path = os.path.join(local_dir, name)
                if _local_matches_remote(dest_path, item):
                    summary["skipped"] += 1
                    if verbose:
                        print(f"  skip           {name}")
                    continue
                if verbose:
                    print(f"  download       {name}")
                _download_blob(drive, item["id"], dest_path)
                summary["downloaded"] += 1
        except HttpError as e:
            summary["errors"] += 1
            print(f"  ERROR on {name}: {e}")

    return summary


# --------------------------------------------------------------------------- #
# Shared parallel-upload machinery (used by BOTH the full-tree sync below and
# push_paths_to_drive in the run_phase1 cell). Defined here so every call site,
# including the early 'sync down/up' convenience cell, can see it.
# --------------------------------------------------------------------------- #
import concurrent.futures, threading, random, time
from googleapiclient.errors import HttpError

DRIVE_UPLOAD_WORKERS = 8       # concurrent Drive transfers; all share ONE service-account
                               # quota bucket, so more than ~8-10 mostly just makes 429s.
_UPLOAD_MAX_TRIES = 6

# googleapiclient's service object (and its underlying httplib2.Http) is NOT safe to
# share across threads. Give each worker thread its own client, built once and cached.
_thread_local = threading.local()
def _thread_drive():
    d = getattr(_thread_local, "drive", None)
    if d is None:
        d = build("drive", "v3", credentials=service_account.Credentials
                  .from_service_account_info(key_info, scopes=SCOPES))
        _thread_local.drive = d
    return d

def _execute_with_backoff(request, what):
    """Run a single Drive API request with exponential backoff + jitter on the errors
    Google tells clients to retry (rate/quota 403, 429, 5xx)."""
    for attempt in range(_UPLOAD_MAX_TRIES):
        try:
            return request.execute()
        except HttpError as e:
            status = getattr(e.resp, "status", None)
            reason = str(e)
            retryable = status in (429, 500, 502, 503, 504) or (
                status == 403 and ("ateLimit" in reason or "uotaExceeded" in reason))
            if not retryable or attempt == _UPLOAD_MAX_TRIES - 1:
                raise
            time.sleep(min(2 ** attempt + random.random(), 32))
    # unreachable

def _upload_one(rel_path, parent_id, local_path, skip_unchanged=True):
    """Worker body: create-or-update a single file under an ALREADY-EXISTING parent.
    Uses this thread's own Drive client. Returns 'uploaded' | 'updated' | 'skipped'.
    With skip_unchanged=True, a remote file whose md5 already matches is left as-is."""
    drv = _thread_drive()
    entry = os.path.basename(rel_path)
    existing = _find_child(drv, parent_id, entry)
    media = MediaFileUpload(local_path, resumable=True)
    if existing:
        if skip_unchanged and existing.get("md5Checksum") == _md5_of(local_path):
            return "skipped"                        # unchanged
        _execute_with_backoff(
            drv.files().update(fileId=existing["id"], media_body=media,
                               supportsAllDrives=True), entry)
        return "updated"
    _execute_with_backoff(
        drv.files().create(body={"name": entry, "parents": [parent_id]},
                           media_body=media, fields="id",
                           supportsAllDrives=True), entry)
    return "uploaded"


# --------------------------------------------------------------------------- #
# Upload side
# --------------------------------------------------------------------------- #

def _ensure_remote_subfolder(drive, parent_id, name):
    """Return the ID of a subfolder named `name` under parent_id, creating it
    if it doesn't exist."""
    existing = _find_child(drive, parent_id, name, mime_type=FOLDER_MIME)
    if existing:
        return existing["id"]
    metadata = {"name": name, "mimeType": FOLDER_MIME, "parents": [parent_id]}
    created = drive.files().create(
        body=metadata, fields="id", supportsAllDrives=True).execute()
    return created["id"]


def sync_local_to_drive_folder(drive, local_dir, folder_id,
                               skip_unchanged=True, verbose=True):
    """Recursively upload local_dir's contents into a Drive folder, IN PARALLEL.

    Same behavior as before -- mirrors local subfolders into Drive, updates same-named
    files in place instead of duplicating, and (with skip_unchanged) skips files whose
    remote md5 already matches -- but the per-file transfers run across a thread pool.

    Mirrors the parallel download: walk the local tree SERIALLY first to enumerate files
    and to ensure+cache every remote subfolder id (so concurrent workers never race to
    create the same folder), then fan out only the create/update calls via _upload_one.

    IMPORTANT -- service account storage quota:
      A service account has NO Drive storage quota of its own. Creating NEW files works
      only when the target folder lives in a SHARED DRIVE. Updating existing files a
      quota-holding user owns is fine.

    Returns {"uploaded", "updated", "skipped", "errors"}.
    """
    # 1) Serial walk: enumerate every local file and ensure its remote parent folder
    #    exists, caching folder ids. All remote-folder mutation happens here, single-
    #    threaded, so there's no race to create the same subfolder twice.
    jobs = []                 # (parent_id, local_path)
    folder_cache = {"": folder_id}
    def _remote_parent_for(rel_dir):
        if rel_dir in folder_cache:
            return folder_cache[rel_dir]
        parent = folder_id
        acc = ""
        for seg in rel_dir.split(os.sep):
            acc = seg if acc == "" else acc + os.sep + seg
            if acc not in folder_cache:
                folder_cache[acc] = _ensure_remote_subfolder(drive, parent, seg)
            parent = folder_cache[acc]
        return parent

    for root, dirs, files in os.walk(local_dir):
        dirs.sort()
        rel_dir = os.path.relpath(root, local_dir)
        rel_dir = "" if rel_dir == "." else rel_dir
        parent_id = _remote_parent_for(rel_dir) if rel_dir else folder_id
        for entry in sorted(files):
            jobs.append((parent_id, os.path.join(root, entry)))

    summary = {"uploaded": 0, "updated": 0, "skipped": 0, "errors": 0}
    if verbose:
        print(f"{len(jobs)} local files to sync up. Uploading with "
              f"{DRIVE_UPLOAD_WORKERS} workers...")

    # 2) Parallel pass: create/update each file via the shared worker. skip_unchanged is
    #    honored inside _upload_one (md5 match -> 'skipped'); when False we still want to
    #    force re-upload, so we pass that through.
    done_n = 0
    with concurrent.futures.ThreadPoolExecutor(max_workers=DRIVE_UPLOAD_WORKERS) as ex:
        futs = {ex.submit(_upload_one, os.path.basename(lp), pid, lp,
                          skip_unchanged): lp
                for (pid, lp) in jobs}
        for fut in concurrent.futures.as_completed(futs):
            lp = futs[fut]
            try:
                res = fut.result()
                summary[res] = summary.get(res, 0) + 1
            except Exception as e:
                summary["errors"] += 1
                reason = str(e)
                if "storageQuotaExceeded" in reason:
                    print(f"  ERROR on {os.path.basename(lp)}: service account has no "
                          f"storage quota; target must be a Shared Drive to create new "
                          f"files. ({reason})")
                else:
                    print(f"  ERROR on {os.path.basename(lp)}: {reason}")
            done_n += 1
            if verbose and done_n % 250 == 0:
                print(f"  {done_n}/{len(jobs)} synced")
    if verbose:
        print(f"Upload complete: {summary}")
    return summary


def _md5_of(path):
    h = hashlib.md5()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


In [ ]:
sync_drive_folder_to_local(drive, CLOUD_DRIVE_BASE, "/content/sample_data")
sync_local_to_drive_folder(drive, "/content/sample_data", CLOUD_DRIVE_BASE)

  skip           mnist_train_small.csv
  skip           mnist_test.csv
  skip           california_housing_train.csv
  skip           california_housing_test.csv
  skip           anscombe.json
  download       manifest.csv
  download       COCHRAN_THAD_05-08-2013_search_v_p000.gif
  download       COCHRAN_THAD_04-12-2013_search_v_p000.gif
  download       COCHRAN_THAD_03-15-2013_search_v_p001.gif
  download       COCHRAN_THAD_03-15-2013_search_v_p000.gif
  download       COCHRAN_THAD_02-14-2013_search_v_p001.gif
  download       COCHRAN_THAD_02-14-2013_search_v_p000.gif
  download       COCHRAN_THAD_01-11-2013_search_v_p001.gif
  download       COCHRAN_THAD_01-11-2013_search_v_p000.gif
  download       CARPER_THOMAS_R_05-02-2013_search_v_p000.gif
  download       CARPER_THOMAS_R_04-05-2013_search_v_p001.gif
  download       CARPER_THOMAS_R_04-05-2013_search_v_p000.gif
  download       CARPER_THOMAS_R_03-07-2013_search_v_p000.gif
  download       CARPER_THOMAS_R_03-05-2013_search_v_p000

In [ ]:
# Helper: map a report to its filing year, so newly downloaded Senate reports land
# directly in reports/<year>/ (matching the per-year layout the corpus now uses on Drive).
# NOTE: bulk reorganization + dedup of the existing Drive corpus is handled by the separate
# `drive_cleanup.ipynb` notebook -- it's intentionally not part of this pipeline anymore.
import re

def year_for_report(date_str, local_path=""):
    """Filing year from the manifest `date` (MM/DD/YYYY); fall back to a year embedded
    in the filename only if the date is missing/unparseable."""
    m = re.search(r"\b(19|20)\d{2}\b", str(date_str))
    if m:
        return m.group(0)
    m = re.search(r"(?:^|[^0-9])((?:19|20)\d{2})(?:[^0-9]|$)", str(local_path))
    return m.group(1) if m else "unknown"


## 2 · Shared helpers — polite HTTP + terms handshake

The site is behind **Akamai bot-manager**, which fingerprints the TLS handshake and
returns `403 $(SERVE_403)` to ordinary Python HTTP clients regardless of headers. We use
**curl_cffi**, which replays a real Chrome TLS fingerprint, so the requests look like a
browser at the transport layer. We still rate-limit ourselves with the delays above.

In [ ]:
!pip -q install curl_cffi

In [ ]:
import time, random
from curl_cffi import requests as cffi_requests   # TLS-impersonating drop-in for requests

def polite_sleep():
    # jitter so we're not a metronome
    time.sleep(REQUEST_DELAY_SECS + random.uniform(0, 0.8))

def _request(session, method, url, **kw):
    """One request: retry transient network errors + 429/5xx; surface hard 4xx blocks."""
    kw.setdefault("timeout", 60)
    last = None
    for attempt in range(MAX_RETRIES):
        try:
            resp = session.request(method, url, **kw)
        except Exception as e:                       # network layer (library-agnostic)
            last = e
            wait = BACKOFF_BASE_SECS * (2 ** attempt)
            print(f"  network error {e!r} -> retry in {wait:.0f}s")
            time.sleep(wait); continue
        if resp.status_code in (429, 500, 502, 503, 504):
            wait = BACKOFF_BASE_SECS * (2 ** attempt)
            print(f"  {resp.status_code} on {url} -> backing off {wait:.0f}s")
            time.sleep(wait); continue
        if resp.status_code in (401, 403):
            snippet = " ".join(resp.text[:300].split())
            raise RuntimeError(
                f"{resp.status_code} blocked at edge for {url} | "
                f"Server={resp.headers.get('Server')} | body: {snippet} | "
                f"Expected curl_cffi to clear Akamai — check impersonate= is set.")
        resp.raise_for_status()
        return resp
    raise RuntimeError(f"Giving up on {url} after {MAX_RETRIES} attempts: {last!r}")

def _csrf(session):
    return session.cookies.get("csrftoken") or session.cookies.get("csrf")

def make_session():
    """Open a TLS-impersonating session and clear the prohibition-agreement gate."""
    s = cffi_requests.Session(impersonate="chrome")   # real Chrome TLS fingerprint
    s.headers.update(BROWSER_HEADERS)
    s.headers["Referer"] = f"{ROOT}/search/"
    # 1) landing page -> csrftoken cookie + agreement form
    _request(s, "GET", f"{ROOT}/search/home/")
    token = _csrf(s)
    # 2) accept the terms
    _request(s, "POST", f"{ROOT}/search/home/",
             data={"csrfmiddlewaretoken": token, "prohibition_agreement": "1"},
             headers={"Referer": f"{ROOT}/search/home/"})
    print("Session established (curl_cffi/Chrome-TLS); terms accepted. CSRF:",
          (_csrf(s) or "")[:8], "...")
    return s

### 2b · Connectivity check

Confirms the TLS-impersonating session gets past Akamai and clears the terms gate. A
printed **200** and an 8-char CSRF token means you're good — go to the smoke test. If you
see a `403 $(SERVE_403)` here, curl_cffi didn't install or `impersonate="chrome"` wasn't
set; re-run section 2.

In [ ]:
_probe = make_session()
_r = _probe.get(f"{ROOT}/search/home/", timeout=60)
print("GET /search/home/ ->", _r.status_code, "| Server:", _r.headers.get("Server"))
assert _r.status_code == 200, "Still blocked — see the note above."
print("Through Akamai. Proceed to the smoke test.")

Session established (curl_cffi/Chrome-TLS); terms accepted. CSRF: hwHO1kIH ...
GET /search/home/ -> 200 | Server: gunicorn
Through Akamai. Proceed to the smoke test.


## 3 · List reports from the JSON endpoint & parse rows

The search grid is a DataTables widget that POSTs to `/search/report/data/` and gets
JSON back. We fetch **all** rows (no server-side type filter, so we never depend on
guessing internal type IDs) and filter **client-side** from each row's own text — robust
against the site changing its option indices.

In [ ]:
import re
from bs4 import BeautifulSoup

LINK_RE = re.compile(r'href="([^"]+)"', re.I)

def _parse_row(row):
    """
    A row is a list of HTML/text cells. Layout (observed):
      [first_name, last_name, office_label, <a>report title</a>, date]
    We parse defensively by *content*, not fixed indices.
    """
    cells = [str(c) for c in row]
    href, title = None, None
    for c in cells:
        m = LINK_RE.search(c)
        if m:
            href = m.group(1)
            title = BeautifulSoup(c, "html.parser").get_text(" ", strip=True)
            break
    text_cells = [BeautifulSoup(c, "html.parser").get_text(" ", strip=True) for c in cells]
    # date = last cell that looks like MM/DD/YYYY
    date = next((t for t in reversed(text_cells) if re.match(r"\d{2}/\d{2}/\d{4}", t)), "")
    office = ""
    for t in text_cells:
        if "senator" in t.lower() or "candidate" in t.lower():
            office = t
    first = text_cells[0] if text_cells else ""
    last  = text_cells[1] if len(text_cells) > 1 else ""
    return {
        "first_name": first, "last_name": last, "office": office,
        "report_title": title or "", "href": href or "", "date": date,
    }

def _keep(rec):
    title = rec["report_title"].lower()
    office = rec["office"].lower()
    if DROP_FILER_HINT in office:                     # drop candidates
        return False
    if KEEP_FILER_HINT not in office and office:      # keep senators (current+former)
        return False
    if "extension" in title:                          # drop Due Date Extension notices
        return False
    return any(k.lower() in title for k in KEEP_REPORT_TYPES)

def _fmt_and_id(href):
    """Classify by URL path: paper=scanned image/PDF; ptr/annual=electronic HTML.
    Kinds can be hyphenated ('extension-notice') and may carry an extra path segment
    before the UUID (e.g. /view/extension-notice/regular/<uuid>/), so parse each part
    independently rather than with one rigid pattern."""
    kind_m = re.search(r"/search/view/([\w-]+)/", href)
    kind = kind_m.group(1) if kind_m else "unknown"
    id_m = re.search(r"[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}", href)
    rid = id_m.group(0) if id_m else href.strip("/").replace("/", "_")
    fmt = "scanned" if kind == "paper" else "electronic_html"  # paper = page-image scans
    return kind, rid, fmt

# Report 'kinds' (first URL path segment) to skip outright — out of scope, and some of
# their view endpoints return 503 with no document behind them.
SKIP_KINDS = {"extension-notice", "blind-trust", "blind_trust"}

def fetch_all_rows(session):
    url = f"{ROOT}/search/report/data/"
    offset, draw, out = 0, 1, []
    while True:
        payload = {
            "draw": str(draw), "start": str(offset), "length": str(PAGE_SIZE),
            "report_types": "[]", "filer_types": "[]",
            "submitted_start_date": f"{DATE_FROM} 00:00:00" if DATE_FROM else "",
            "submitted_end_date":  f"{DATE_TO} 23:59:59" if DATE_TO else "",
            "candidate_state": "", "senator_state": "", "office_id": "",
            "first_name": "", "last_name": "",
        }
        headers = {"X-Requested-With": "XMLHttpRequest",
                   "X-CSRFToken": _csrf(session),
                   "Referer": f"{ROOT}/search/"}
        resp = _request(session, "POST", url, data=payload, headers=headers)
        rows = resp.json().get("data", [])
        if not rows:
            break
        out.extend(rows)
        print(f"  fetched {len(out)} rows...", end="\r")
        offset += PAGE_SIZE; draw += 1
        polite_sleep()
    print(f"\nTotal listing rows: {len(out)}")
    return out

def build_index(rows):
    recs = []
    for r in rows:
        rec = _parse_row(r)
        if not rec["href"] or not _keep(rec):
            continue
        kind, rid, fmt = _fmt_and_id(rec["href"])
        if kind in SKIP_KINDS:            # extension notices etc. — never request them
            continue
        rec.update(kind=kind, report_id=rid, format=fmt)
        recs.append(rec)
    print(f"Kept {len(recs)} Annual/Periodic senator reports after filtering.")
    return recs

## 4 · Manifest (resume state)

In [ ]:
import pandas as pd
import sqlite3

MANIFEST_COLS = ["report_id","first_name","last_name","office","report_title",
                 "date","href","kind","format","local_path","ocr_path","ocr_status"]

# --- SQLite state layer -----------------------------------------------------
# The manifest's in-flight home is a LOCAL SQLite DB (fast, atomic per-row updates,
# crash-safe within a session). The CSV remains the durable, portable contract that
# Phase 2 and the downstream pipeline read -- we export it at every checkpoint.
#
# Why local: the DB must sit on real local disk, never the Drive round-trip, for
# SQLite's locking/WAL to behave. It's ephemeral scaffolding -- if Colab restarts and
# the DB is gone, load_manifest() rebuilds it from the last CSV we pushed to Drive.
#
# load_manifest()/save_manifest()/upsert() keep their original signatures (DataFrame
# in, DataFrame out) so nothing downstream changes; only the storage underneath does.

MANIFEST_DB = "/content/manifest.db"    # local ephemeral disk, NOT the Drive path

def _connect():
    """One connection per call/thread. WAL + busy_timeout so concurrent Drive-upload
    workers reading state don't trip over the single writer."""
    conn = sqlite3.connect(MANIFEST_DB, timeout=30)
    conn.execute("PRAGMA journal_mode=WAL;")
    conn.execute("PRAGMA busy_timeout=30000;")   # wait up to 30s rather than raising BUSY
    conn.execute("PRAGMA synchronous=NORMAL;")   # safe with WAL; fewer fsyncs
    return conn

def _ensure_db():
    """Create the table if missing; if the DB is empty/new but a CSV exists (e.g. we
    resumed after the ephemeral disk was wiped), rebuild the DB from that CSV."""
    conn = _connect()
    cols_sql = ", ".join(f'"{c}" TEXT' for c in MANIFEST_COLS)
    conn.execute(f'CREATE TABLE IF NOT EXISTS reports ({cols_sql}, '
                 f'PRIMARY KEY ("report_id"));')
    conn.commit()
    (n,) = conn.execute("SELECT COUNT(*) FROM reports;").fetchone()
    if n == 0 and os.path.exists(MANIFEST_CSV):
        df = pd.read_csv(MANIFEST_CSV, dtype=str).fillna("")
        for c in MANIFEST_COLS:
            if c not in df.columns:
                df[c] = ""
        df[MANIFEST_COLS].to_sql("reports", conn, if_exists="append", index=False)
        conn.commit()
        print(f"Rebuilt manifest DB from CSV ({len(df)} rows).")
    conn.close()

def load_manifest():
    """Return the manifest as a DataFrame (same shape as before). Reads from the local
    DB, rebuilding it from CSV first if this is a fresh/wiped session."""
    _ensure_db()
    conn = _connect()
    df = pd.read_sql_query("SELECT * FROM reports;", conn).fillna("")
    conn.close()
    for c in MANIFEST_COLS:
        if c not in df.columns:
            df[c] = ""
    return df[MANIFEST_COLS] if len(df) else pd.DataFrame(columns=MANIFEST_COLS)

def save_manifest(df):
    """Checkpoint: persist the DataFrame into the local DB, then export the durable CSV
    to the Drive-synced path. Single-writer -- call this from the owning thread only."""
    conn = _connect()
    with conn:                                   # atomic transaction
        conn.execute("DELETE FROM reports;")
        df[MANIFEST_COLS].to_sql("reports", conn, if_exists="append", index=False)
    conn.close()
    # Export the portable contract the rest of the pipeline reads.
    tmp = MANIFEST_CSV + ".tmp"
    df.to_csv(tmp, index=False); os.replace(tmp, MANIFEST_CSV)

def upsert(df, rec):
    """Insert rec if its report_id is new; return (df, is_new). Operates on the in-memory
    DataFrame exactly as before -- persistence happens at save_manifest checkpoints."""
    if (df["report_id"] == rec["report_id"]).any():
        return df, False
    row = {c: rec.get(c, "") for c in MANIFEST_COLS}
    row["ocr_status"] = "pending" if str(rec.get("format","")).startswith("scanned") else "n/a"
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    return df, True


## 5 · Download a report

Electronic reports are saved as `.html` (already machine-readable). Paper filings are
scanned — the `paper` viewer may hand back a PDF directly or an HTML page embedding page
images; we handle both and save whatever it yields. The smoke test shows you exactly what
comes back for a paper report so you can confirm this branch.

In [ ]:
import mimetypes

def _safe(s):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", s).strip("_")[:80]

def _dest_dir_for(rec):
    """Where this report's files should be saved: reports/<year>/ when we can
    determine the filing year, else reports/. Matches the per-year layout (see drive_cleanup.ipynb) so
    fresh downloads land in the same place a re-organize would put them."""
    try:
        year = year_for_report(rec.get("date", ""))
    except NameError:
        # foldering cell not run in this session; fall back to flat CORPUS_DIR
        return CORPUS_DIR
    d = os.path.join(CORPUS_DIR, year)
    os.makedirs(d, exist_ok=True)
    return d

def download_report(session, rec):
    dest_dir = _dest_dir_for(rec)
    url = ROOT + rec["href"] if rec["href"].startswith("/") else rec["href"]
    base = _safe(f'{rec["last_name"]}_{rec["first_name"]}_{rec["date"].replace("/","-")}_{rec["report_id"][:8]}')
    resp = _request(session, "GET", url)
    ctype = resp.headers.get("Content-Type", "").lower()

    if rec["format"] == "electronic_html":
        path = os.path.join(dest_dir, base + ".html")
        with open(path, "wb") as f: f.write(resp.content)
        return [path]

    # paper / scanned: a direct PDF (rare) or an HTML viewer that embeds each page as an
    # <img class="filingImage"> GIF hosted on efd-media-public.senate.gov.
    if "application/pdf" in ctype or url.lower().endswith(".pdf"):
        path = os.path.join(dest_dir, base + ".pdf")
        with open(path, "wb") as f: f.write(resp.content)
        return [path]

    soup = BeautifulSoup(resp.text, "html.parser")
    saved = []
    # primary path: the scanned page images (GIFs), selected by their class
    srcs = [im.get("src") for im in soup.select("img.filingImage") if im.get("src")]
    seen = set()
    srcs = [u for u in srcs if not (u in seen or seen.add(u))]   # de-dup, keep page order
    for i, src in enumerate(srcs):
        u = src if src.startswith("http") else ROOT + src
        r2 = _request(session, "GET", u); polite_sleep()
        ext = os.path.splitext(u.split("?")[0])[1].lower() or ".gif"
        p = os.path.join(dest_dir, f"{base}_p{i:03d}{ext}")
        with open(p, "wb") as f: f.write(r2.content)
        saved.append(p)
    # fallback: an embedded/linked PDF
    if not saved:
        pdf_links = [a["href"] for a in soup.find_all("a", href=True)
                     if a["href"].lower().endswith(".pdf")]
        for i, href in enumerate(pdf_links):
            u = href if href.startswith("http") else ROOT + href
            r2 = _request(session, "GET", u); polite_sleep()
            p = os.path.join(dest_dir, f"{base}_{i}.pdf")
            with open(p, "wb") as f: f.write(r2.content)
            saved.append(p)
    if not saved:  # unrecognized structure -> keep raw HTML so we can inspect it
        p = os.path.join(dest_dir, base + "_viewer.html")
        with open(p, "wb") as f: f.write(resp.content)
        saved.append(p)
    return saved

## 6 · 🔎 Smoke test — RUN THIS FIRST

Confirms the handshake + payload return real rows, prints a couple of parsed records, and
downloads **one electronic** and **one paper** report so you can eyeball the output before
the full sweep. If this fails or returns 0 rows, stop and inspect the printed raw response
rather than launching the bulk job.

In [ ]:
_s = make_session()
_rows = []
# just the first page for the smoke test:
import json as _json
_payload = {"draw":"1","start":"0","length":"25","report_types":"[]","filer_types":"[]",
            "submitted_start_date": f"{DATE_FROM} 00:00:00" if DATE_FROM else "",
            "submitted_end_date":"", "candidate_state":"","senator_state":"","office_id":"",
            "first_name":"","last_name":""}
_r = _request(_s, "POST", f"{ROOT}/search/report/data/", data=_payload,
              headers={"X-Requested-With":"XMLHttpRequest","X-CSRFToken":_csrf(_s),
                       "Referer": f"{ROOT}/search/"})
_data = _r.json()
print("Keys in response:", list(_data.keys()))
_rows = _data.get("data", [])
print("Rows on page 1:", len(_rows))
assert _rows, "No rows! Inspect _data below; the payload field names may have changed."
_recs = build_index(_rows)
for _rec in _recs[:3]:
    print(" •", _rec["last_name"], _rec["first_name"], "|", _rec["report_title"], "|", _rec["format"])

# try one of each format if present on this page
for _fmt in ("electronic_html", "scanned_pdf"):
    _hit = next((r for r in _recs if r["format"] == _fmt), None)
    if _hit:
        _paths = download_report(_s, _hit); polite_sleep()
        print(f"  downloaded {_fmt}: {[os.path.basename(p) for p in _paths]}")
    else:
        print(f"  (no {_fmt} on page 1 — will appear in full run)")

Session established (curl_cffi/Chrome-TLS); terms accepted. CSRF: FFwuko8r ...
Keys in response: ['draw', 'recordsTotal', 'recordsFiltered', 'data', 'result']
Rows on page 1: 25
Kept 12 Annual/Periodic senator reports after filtering.
 • Tuberville Thomas H | Periodic Transaction Report for 07/16/2026 | electronic_html
 • Cassidy William | Annual Report for CY 2025 | electronic_html
 • Curtis John R | Annual Report for CY 2025 | electronic_html
  downloaded electronic_html: ['Tuberville_Thomas_H_07-16-2026_392ac3e5.html']
  (no scanned_pdf on page 1 — will appear in full run)


## 7 · Phase 1 — full collection (resumable)

In [ ]:
# --- Incremental push (parallel): upload only the files this run actually wrote. ---
# The old code re-walked the whole tree every 100 records and hashed thousands of
# already-uploaded files each time. Instead we track the local paths we just wrote and
# push exactly those. Uploads are network-I/O-bound, so we fan them out across a small
# thread pool -- this parallelism is on the GOOGLE DRIVE side only; the Senate fetches
# in run_phase1 stay serial and paced (we're slow there on purpose).
# (Parallel-upload workers _thread_drive/_execute_with_backoff/_upload_one and
#  the DRIVE_UPLOAD_WORKERS setting are defined in the sync-helpers cell above,
#  so both push_paths_to_drive and the full-tree sync share one tested worker.)

def push_paths_to_drive(paths):
    """Upload a specific set of local files to their mirrored Drive location, in parallel.

    Signature unchanged: takes a list of local paths, returns the count pushed.

    Subfolders are resolved SERIALLY on this (main) thread first, so concurrent workers
    never race to create the same reports/<year>/ folder (which would make duplicates).
    Only the file-level create/update -- which is naturally per-file -- is parallelized.
    """
    # De-dup and keep only real files.
    todo = []
    seen = set()
    for p in paths:
        if p and p not in seen and os.path.exists(p):
            seen.add(p); todo.append(p)
    if not todo:
        return 0

    # 1) Serial pass: ensure every needed remote folder exists, cache its id. Cheap
    #    metadata calls, and doing them here removes the only shared-write race.
    parent_of = {}
    folder_id = {}                                   # rel-dir tuple -> Drive folder id
    for p in todo:
        rel = os.path.relpath(p, "/content/sample_data")
        parts = rel.split(os.sep)
        parent = CLOUD_DRIVE_BASE
        acc = ()
        for seg in parts[:-1]:
            acc = acc + (seg,)
            if acc not in folder_id:
                folder_id[acc] = _ensure_remote_subfolder(drive, parent, seg)
            parent = folder_id[acc]
        parent_of[p] = (rel, parent)

    # 2) Parallel pass: file create/update across the worker pool.
    pushed = 0
    errors = 0
    with concurrent.futures.ThreadPoolExecutor(max_workers=DRIVE_UPLOAD_WORKERS) as ex:
        futs = {ex.submit(_upload_one, parent_of[p][0], parent_of[p][1], p): p
                for p in todo}
        for fut in concurrent.futures.as_completed(futs):
            p = futs[fut]
            try:
                res = fut.result()
                if res in ("uploaded", "updated"):
                    pushed += 1
            except Exception as e:
                errors += 1
                print(f"  push ERROR on {os.path.basename(p)}: {e}")
    if errors:
        print(f"  ({errors} upload(s) failed after retries; rows remain in manifest "
              f"for the next sync to retry.)")
    return pushed

def run_phase1():
    session = make_session()
    rows = fetch_all_rows(session)
    recs = build_index(rows)

    df = load_manifest()
    # A report is "done" only if it produced real files. Paper reports that fell back to a
    # *_viewer.html (older buggy runs) are NOT done — re-fetch them for their page images.
    def _complete(lp):
        return bool(lp) and "_viewer.html" not in lp
    done_ids = set(df.loc[df["local_path"].map(_complete), "report_id"])
    print(f"Manifest has {len(df)} rows; {len(done_ids)} already complete.")

    new = 0
    pending_push = []          # local files written since the last Drive push
    for i, rec in enumerate(recs, 1):
        df, is_new = upsert(df, rec)
        # keep the format label current for pre-existing rows (e.g. scanned_pdf -> scanned)
        df.loc[df["report_id"] == rec["report_id"], "format"] = rec["format"]
        if rec["report_id"] in done_ids:
            continue
        try:
            paths = download_report(session, rec)
            df.loc[df["report_id"] == rec["report_id"], "local_path"] = ";".join(paths)
            pending_push.extend(paths)      # remember to push just these
            new += 1
        except Exception as e:
            print(f"  FAILED {rec['report_id']}: {e}")
            df.loc[df["report_id"] == rec["report_id"], "ocr_status"] = "download_failed"
        if i % 25 == 0:
            save_manifest(df); print(f"  {i}/{len(recs)} processed, {new} new downloads")
        if i % 100 == 0 and pending_push:
            # Push ONLY the manifest.csv plus the files written since the last push.
            n = push_paths_to_drive([MANIFEST_CSV] + pending_push)
            print(f"  synced {n} changed files to Drive")
            pending_push = []
        polite_sleep()

    save_manifest(df)
    if pending_push:                        # flush any tail (< 100) at the end
        push_paths_to_drive([MANIFEST_CSV] + pending_push)
    print(f"Phase 1 done. {new} new files this run. Manifest: {MANIFEST_CSV}")

run_phase1()

Session established (curl_cffi/Chrome-TLS); terms accepted. CSRF: Np4KnDHj ...

Total listing rows: 5571
Kept 4249 Annual/Periodic senator reports after filtering.
Manifest has 3599 rows; 3500 already complete.
  3300/4249 processed, 47 new downloads
  3325/4249 processed, 51 new downloads
  3425/4249 processed, 69 new downloads
  3525/4249 processed, 82 new downloads
  3600/4249 processed, 99 new downloads
  3625/4249 processed, 124 new downloads
  3650/4249 processed, 149 new downloads
  3675/4249 processed, 174 new downloads
  3700/4249 processed, 199 new downloads
  ERROR on Cardin_Benjamin_L_05-15-2015_fcc32702.html: <HttpError 500 when requesting https://www.googleapis.com/drive/v3/files?q=name+%3D+%27Cardin_Benjamin_L_05-15-2015_fcc32702.html%27+and+%271gTwKlQvyxGnYLSZmXWLGgwa57dvl_TOx%27+in+parents+and+trashed+%3D+false&fields=files%28id%2C+name%2C+mimeType%2C+size%2C+modifiedTime%2C+md5Checksum%29&pageSize=10&supportsAllDrives=true&includeItemsFromAllDrives=true&alt=json retur

In [ ]:
sync_local_to_drive_folder(drive, "/content/sample_data", CLOUD_DRIVE_BASE)

## 8 · Phase 2 — OCR the scanned reports with Gemma 4 12B

> **Switch the runtime to an L4 GPU now** (*Runtime → Change runtime type → L4 GPU*),
> then run the cells below. Electronic HTML reports are already machine-readable and are
> skipped here — only `scanned` rows (paper filings saved as page-image GIFs) get OCR'd.

In [ ]:
sync_drive_folder_to_local(drive, CLOUD_DRIVE_BASE, "/content/sample_data")
sync_local_to_drive_folder(drive, "/content/sample_data", CLOUD_DRIVE_BASE)

In [ ]:
# One-time installs for the GPU runtime:
!pip -q install -U "transformers>=4.44" accelerate bitsandbytes pymupdf pillow

In [ ]:
# Authenticate to Hugging Face (Gemma is license-gated).
# Accept the license once at https://huggingface.co/google/gemma-4-12b-it
from huggingface_hub import login
login()  # paste a token with 'read' access, or set it in Colab Secrets as HF_TOKEN

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

quant = None
if LOAD_IN_4BIT:
    quant = BitsAndBytesConfig(load_in_4bit=True,
                               bnb_4bit_compute_dtype=torch.bfloat16,
                               bnb_4bit_quant_type="nf4")

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,           # keep host RAM low during load
    quantization_config=quant,
)
model.eval()
print("Model loaded on:", next(model.parameters()).device)

In [ ]:
import io, json, os, fitz  # PyMuPDF
from PIL import Image

IMAGE_EXTS = (".gif", ".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")

def _page_images(path, dpi):
    """Yield PIL RGB page images: rasterize PDFs; load scan images (GIF/PNG/...) directly."""
    ext = os.path.splitext(path)[1].lower()
    if ext == ".pdf":
        doc = fitz.open(path)
        for i in range(doc.page_count):
            pix = doc[i].get_pixmap(dpi=dpi)
            yield Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
    elif ext in IMAGE_EXTS:
        yield Image.open(path).convert("RGB")
    # anything else (e.g. a stray _viewer.html) is skipped

def ocr_image(img):
    msgs = [{"role": "user", "content": [
        {"type": "image", "image": img},
        {"type": "text",  "text": OCR_PROMPT},
    ]}]
    inputs = processor.apply_chat_template(
        msgs, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt").to(model.device)
    in_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    return processor.decode(out[0][in_len:], skip_special_tokens=True).strip()

def ocr_document(paths):
    """OCR every page across all saved files for one report (multi-GIF scans or a PDF)."""
    pages = []
    for p in paths:
        for img in _page_images(p, PDF_RENDER_DPI):
            pages.append({"page": len(pages) + 1, "text": ocr_image(img)})
            print(f"    page {len(pages)} ok", end="\r")
    return pages

In [ ]:
def run_phase2():
    df = load_manifest()
    todo = df[df["format"].astype(str).str.startswith("scanned") & (df["ocr_status"] != "done")]
    print(f"{len(todo)} scanned documents to OCR.")

    for n, (_, row) in enumerate(todo.iterrows(), 1):
        rid = row["report_id"]
        out_path = os.path.join(OCR_DIR, f"{rid}.json")
        if os.path.exists(out_path):                      # resume: already OCR'd
            df.loc[df["report_id"] == rid, ["ocr_path","ocr_status"]] = [out_path, "done"]
            continue
        # a report may have produced several files (multi-page GIF scans, or a PDF)
        files = [p for p in str(row["local_path"]).split(";")
                 if os.path.splitext(p)[1].lower() in (".pdf",) + IMAGE_EXTS]
        if not files:
            df.loc[df["report_id"] == rid, "ocr_status"] = "no_scan_files"; continue
        try:
            pages = ocr_document(files)
            record = {"report_id": rid,
                      "filer": f'{row["first_name"]} {row["last_name"]}',
                      "report_title": row["report_title"], "date": row["date"],
                      "model": MODEL_ID, "dpi": PDF_RENDER_DPI,
                      "pages": pages,
                      "full_text": "\n\n".join(p["text"] for p in pages)}
            with open(out_path, "w") as f: json.dump(record, f, ensure_ascii=False, indent=2)
            df.loc[df["report_id"] == rid, ["ocr_path","ocr_status"]] = [out_path, "done"]
        except Exception as e:
            print(f"  FAILED {rid}: {e}")
            df.loc[df["report_id"] == rid, "ocr_status"] = "ocr_failed"
        save_manifest(df)                                  # checkpoint every document
        print(f"  {n}/{len(todo)} done ({rid[:8]})")
    print("Phase 2 complete.")

run_phase2()

## 9 · Where your data landed
- **`reports/`** — every downloaded report (`.html` = electronic, `.pdf`/images = scanned)
- **`ocr/`** — one JSON per scanned document (`pages[]` + `full_text`)
- **`manifest.csv`** — the index; `format` tells you electronic vs scanned, `ocr_status`
  tracks OCR progress. Your downstream Claude Code pipeline can read this directly.

Re-running either phase resumes where it left off. To force a re-download or re-OCR of a
document, delete its file(s) and clear its row's `local_path` / `ocr_status`.